# Iteration 3: Full Autism Pathway Model

This notebook builds a discrete-event simulation of an autism assessment pathway. It models how referrals arrive, move through clinical decision points, wait for appointment capacity, and leave the pathway through rejection, discharge, diagnosis, or self-removal.

## Model Summary

The pathway is represented as a sequence of patient-level events rather than as a spreadsheet average. Each simulated patient follows the same broad route, but random sampling determines arrivals, activity times, and branching decisions. This lets the model estimate not only total activity, but also waiting times, backlogs, resource use, and variation between replications.

The main pathway stages are:

1. Referral and triage decision.
2. Screening appointment.
3. Pre-assessment appointment.
4. Assessment appointment.
5. Further assessment appointment where needed.
6. Post-diagnostic clinical or other support.
7. Review, discharge, or self-removal from the pathway.

## Resource Modelling Approach

The notebook uses SimPy for discrete-event simulation and a custom `CalendarAwareQueueResource` for service capacity. Instead of assuming staff are continuously available, each clinical stage releases a fixed number of appointment slots on weekdays. Patients queue until a slot is released for their stage.

The resource model includes:

1. Weekday-only arrivals and weekday-only appointment capacity.
2. Separate slot capacities for screening, pre-assessment, assessment, further assessment, post-diagnostic clinical support, post-diagnostic other support, and review.
3. Queue length tracking at each stage.
4. Released, used, and unused slot counts.
5. Utilisation and backlog metrics for operational reporting.

## How To Read This Notebook

The notebook is organised from model setup through execution and verification. The early sections define assumptions and parameters, the middle sections define patient flow and resources, and the final sections run tests, replications, and verification and validation checks.


## 1. Imports

This cell loads the Python libraries needed by the notebook. `simpy` provides the discrete-event simulation engine, `numpy` and `pandas` support numerical analysis and tabular output, and `joblib` allows multiple replications to run in parallel. The custom distribution classes are imported from the project package so that arrival times, service durations, and branching decisions can be sampled consistently.


In [69]:
import itertools
import math
import pandas as pd
from collections import deque
import numpy as np
import simpy
from joblib import Parallel, delayed
import copy

from adhd_simpy.Model.distributions import (
    Exponential,
    Triangular,
    Bernoulli,
)


## 2. Global Settings and Model Parameters

This section contains the core parameters used to configure the simulation model during development and testing. These include the simulation horizon, referral arrival rates, activity durations, pathway branching probabilities, appointment capacities, and random number control settings.

The parameter values defined here are synthetic modelling assumptions used to develop, test, and validate the model logic and patient flow structure. They should not be interpreted as NHS operational data, service performance measures, staffing levels, waiting-time targets, or clinically validated estimates.

Keeping these parameters in a single location improves transparency and reproducibility. Alternative scenarios can be explored by modifying these assumptions and rerunning the simulation while preserving the underlying model structure.


In [ ]:
# ============================================================
# GLOBAL SETTINGS & CONFIGURATIONS
# ============================================================
TRACE = False
RUN_LENGTH = 365 * 5  # Active simulation horizon (5 Years in Days)
REFERRALS_PER_DAY = 5  # Expected total incoming volume across a full week

# Arrival Calculation Adjustment:

IAT_WEEKDAY = 1 / REFERRALS_PER_DAY  # Weekday inter-arrival time

# Clinical task durations in hours: optimistic, likely, pessimistic.
DURATION_SCREENING = [0.5, 0.75, 1.5]
DURATION_PRE_ASSESSMENT = [1.0, 2.0, 3.0]
DURATION_ASSESSMENT = [2.0, 3.0, 5.0]
DURATION_FURTHER_ASSESSMENT = [1.5, 2.5, 4.0]
DURATION_POST_DIAG_CLINICAL = [1.0, 1.5, 2.5]
DURATION_POST_DIAG_OTHER = [1.5, 2.5, 4.0]
DURATION_REVIEW = [0.25, 0.5, 1.0]

# Pathway Branching Ratios (0.0 to 1.0)
PCT_REFERRAL_REJECTED = 0.25
PCT_SCREENING_DISCHARGED = 0.12
PCT_PRE_ASS_REJECTED = 0.08
PCT_NON_DIAGNOSIS_AT_ASSESSMENT = 0.15
PCT_NON_DIAGNOSIS_AT_FURTHER_ASSESSMENT = 0.10
PCT_POST_DIAG_CLINICAL = 0.72
PCT_REMOVAL_DISCHARGE = 0.88

# Fixed consultation slots released per weekday.
SCREENING_SLOT = 7
PRE_ASSESSMENT_SLOT = 3
ASSESSMENT_SLOT = 5
FURTHER_ASSESSMENT_SLOT = 3
POST_DIAG_CLINICAL_SLOT = 4
POST_DIAG_OTHER_SLOT = 2
REVIEW_SLOT = 2

N_STREAMS = 15  # Number of independent pseudo-random sub-streams per run
DEFAULT_RND_SET = 42  # Starting baseline random seed
N_REP = 20  # Total number of cross-validation replications to execute


## 3. Trace Utility

The trace helper is a simple switch for detailed event logging. When `TRACE` is set to `True`, the simulation prints patient-level movement through the pathway. When it is `False`, the model runs quietly and only reports summary results.

This is useful for debugging because the same model code can be inspected at event level without adding print statements throughout the patient pathway logic.


In [71]:
def trace(msg: str) -> None:
    """
    Conditionally print an event-level trace message.

    Parameters
    ----------
    msg : str
        Message describing a simulation event.

    Returns
    -------
    None
        The function prints only when global tracing is enabled.
    """
    if TRACE:
        print(msg)


## 4. Audit Class

The `Audit` class collects model evidence while the simulation runs. It records referral-to-treatment waiting times, sampled queue lengths, and resource statistics. These data are then converted into summary KPIs at the end of each run.

This separates measurement from pathway logic: the patient flow code focuses on what happens to patients, while the audit object focuses on what should be counted and reported.


In [72]:
#======================================================
# AUDIT CLASS FOR KPI COLLECTION
#======================================================
class Audit:
    """
    Collect run-level waiting time, queue, and resource metrics.

    Attributes
    ----------
    STAGE_NAMES : list of str
        Ordered names of the pathway resources tracked by the audit.
    """

    STAGE_NAMES = [
        "screening",
        "pre_assessment",
        "assessment",
        "further_assessment",
        "post_diag_clinical",
        "post_diag_other",
        "review",
    ]

    def __init__(self):
        """
        Initialise an empty audit object.

        Returns
        -------
        None
            The object is reset during construction.
        """
        self.reset()

    def reset(self):
        """
        Clear all waiting time, queue, and resource measurements.

        Returns
        -------
        None
            Internal audit stores are replaced with empty containers.
        """
        self.rtt_days = {
            "screening": [],
            "pre_assessment": [],
            "assessment": [],
            "further_assessment": [],
            "diagnosis": [],
        }

        self.queue_lengths = {stage: [] for stage in self.STAGE_NAMES}

        self.resource_stats = {}

    def record_rtt(self, metric_name, value_days):
        """
        Record a referral-to-treatment time observation.

        Parameters
        ----------
        metric_name : str
            Name of the waiting-time metric to update.
        value_days : float
            Observed elapsed time in simulated days.

        Returns
        -------
        None
            The value is appended when the metric name is recognised.
        """
        if metric_name in self.rtt_days:
            self.rtt_days[metric_name].append(float(value_days))

    def record_queue_length(self, stage_name, queue_length):
        """
        Record an observed queue length for a pathway stage.

        Parameters
        ----------
        stage_name : str
            Name of the resource queue being measured.
        queue_length : int or float
            Number of patients waiting at the measurement time.

        Returns
        -------
        None
            The value is appended when the stage name is recognised.
        """
        if stage_name in self.queue_lengths:
            self.queue_lengths[stage_name].append(float(queue_length))

    def capture_resource_stats(self, stage_name, resource):
        """
        Capture summary statistics from a resource object.

        Parameters
        ----------
        stage_name : str
            Name used to store the resource summary.
        resource : CalendarAwareQueueResource
            Resource exposing a ``get_summary`` method.

        Returns
        -------
        None
            Resource statistics are stored on the audit object.
        """
        summary = resource.get_summary()

        self.resource_stats[stage_name] = {
            "utilisation": float(summary.get("utilisation_pct", 0)) / 100.0,
            "slots_used": float(summary.get("used_slots", 0)),
            "slots_released": float(summary.get("released_slots", 0)),
            "max_queue_length": float(summary.get("max_queue_length", 0)),
            "queue_backlog": float(summary.get("queue_backlog", 0)),
        }

    def summarize(self, flow_results, run_length):
        """
        Build final KPI values for one simulation run.

        Parameters
        ----------
        flow_results : dict
            Counter dictionary produced by the patient pathway logic.
        run_length : float
            Simulation horizon in days.

        Returns
        -------
        dict
            Summary waiting-time, queue, utilisation, and flow KPIs.
        """
        resource_stats = list(self.resource_stats.values())

        total_slots_used = sum(
            stat.get("slots_used", 0) for stat in resource_stats
        )

        total_slots_released = sum(
            stat.get("slots_released", 0) for stat in resource_stats
        )

        queue_backlog_total = sum(
            stat.get("queue_backlog", 0) for stat in resource_stats
        )

        system_utilization = (
            (total_slots_used / total_slots_released * 100)
            if total_slots_released > 0
            else 0.0
        )

        def get_rtt_mean(stage):
            """Return the mean RTT for a stage.

            Parameters
            ----------
            stage : str
                Name of the RTT list to summarise.

            Returns
            -------
            float
                Mean waiting time in days, or zero if empty.
            """

            data = self.rtt_days.get(stage, [])

            return float(np.mean(data)) if len(data) > 0 else 0.0

        summary = {
            "ACCESS_REFERRAL_TO_SCREENING_RTT_DAYS": get_rtt_mean(
                "screening"
            ),
            "ACCESS_REFERRAL_TO_PRE_ASSESSMENT_RTT_DAYS": get_rtt_mean(
                "pre_assessment"
            ),
            "ACCESS_REFERRAL_TO_ASSESSMENT_RTT_DAYS": get_rtt_mean(
                "assessment"
            ),
            "ACCESS_REFERRAL_TO_FURTHER_ASSESSMENT_RTT_DAYS": get_rtt_mean(
                "further_assessment"
            ),
            "ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS": get_rtt_mean(
                "diagnosis"
            ),
            "QUEUE_TOTAL_BACKLOG": float(queue_backlog_total),
            "CAPACITY_TOTAL_SLOT_USAGE": float(total_slots_used),
            "CAPACITY_TOTAL_SLOT_RELEASED": float(total_slots_released),
            "OVERALL_SYSTEM_UTILISATION": float(system_utilization),
            "FLOW_DIAGNOSIS_RATE_PCT": (
                100.0
                * flow_results.get("FLOW_DIAGNOSIS_CONFIRMED", 0)
                / max(flow_results.get("ARRIVED_TOTAL", 1), 1)
            ),
        }

        for stage in self.STAGE_NAMES:

            queue_values = self.queue_lengths.get(stage, [])
            stat = self.resource_stats.get(stage, {})

            summary[f"QUEUE_MEAN_{stage.upper()}"] = (
                float(np.mean(queue_values)) if queue_values else 0.0
            )

            summary[f"CAPACITY_UTILISATION_{stage.upper()}"] = stat.get(
                "utilisation", 0.0
            )

        return summary


## 5. Experiment Setup

The `Experiment` class is the main parameter container for a simulation run. It stores demand assumptions, branching probabilities, activity duration inputs, resource capacities, and random stream settings.

It also builds the result counter dictionary used throughout the model. Because each replication receives its own `Experiment` object, runs can be repeated, compared, and tested without results leaking between replications.


In [73]:
# ============================================================
# EXPERIMENT SETUP PARAMETERS
# ============================================================

class Experiment:
    """
    Store parameters and random streams for one experiment.

    Parameters
    ----------
    auditor : Audit
        Audit object used to collect run measurements.
    random_number_set : int, default DEFAULT_RND_SET
        Base seed used when deterministic sampling is enabled.
    n_streams : int, default N_STREAMS
        Number of independent random streams to create.
    use_fixed_seed : bool, default True
        Whether to use deterministic stream generation.
    iat : float, default IAT_WEEKDAY
        Mean inter-arrival time in simulated days.
    **kwargs
        Optional overrides for probabilities, durations, and capacities.
    """

    def __init__(
        self,
        auditor,
        random_number_set=DEFAULT_RND_SET,
        n_streams=N_STREAMS,
        use_fixed_seed=True,
        iat=IAT_WEEKDAY,
        **kwargs,
    ):
        """
        Initialise experiment parameters and stochastic sampling.

        Parameters
        ----------
        auditor : Audit
            Audit object used to collect run measurements.
        random_number_set : int, default DEFAULT_RND_SET
            Base seed used when deterministic sampling is enabled.
        n_streams : int, default N_STREAMS
            Number of independent random streams to create.
        use_fixed_seed : bool, default True
            Whether to use deterministic random sampling.
        iat : float, default IAT_WEEKDAY
            Mean weekday inter-arrival time in days.
        **kwargs
            Optional parameter overrides for scenario testing.

        Returns
        -------
        None
            The experiment object is configured in place.
        """
        self.auditor = auditor
        self.random_number_set = random_number_set
        self.base_random_number_set = random_number_set
        self.n_streams = n_streams
        self.use_fixed_seed = use_fixed_seed
        self.iat = iat

        # Branching probabilities
        self.triage_rejected = kwargs.get(
            "triage_rejected", PCT_REFERRAL_REJECTED
        )
        self.screening_discharge = kwargs.get(
            "screening_discharge", PCT_SCREENING_DISCHARGED
        )
        self.pre_assessment_rejection = kwargs.get(
            "pre_assessment_rejection", PCT_PRE_ASS_REJECTED
        )
        self.pct_non_diag_at_assessment = kwargs.get(
            "pct_non_diag_at_assessment", PCT_NON_DIAGNOSIS_AT_ASSESSMENT
        )
        self.pct_non_diag_at_further_assessment = kwargs.get(
            "pct_non_diag_at_further_assessment",
            PCT_NON_DIAGNOSIS_AT_FURTHER_ASSESSMENT,
        )
        self.pct_post_diag_clinical = kwargs.get(
            "pct_post_diag_clinical", PCT_POST_DIAG_CLINICAL
        )
        self.pct_removal_discharge = kwargs.get(
            "pct_removal_discharge", PCT_REMOVAL_DISCHARGE
        )

        # Task durations
        self.dur_screening = kwargs.get("dur_screening", DURATION_SCREENING)
        self.dur_pre_assessment = kwargs.get(
            "dur_pre_assessment", DURATION_PRE_ASSESSMENT
        )
        self.dur_assessment = kwargs.get(
            "dur_assessment", DURATION_ASSESSMENT
        )
        self.dur_further_assessment = kwargs.get(
            "dur_further_assessment", DURATION_FURTHER_ASSESSMENT
        )
        self.dur_post_diag_clinical = kwargs.get(
            "dur_post_diag_clinical", DURATION_POST_DIAG_CLINICAL
        )
        self.dur_post_diag_other = kwargs.get(
            "dur_post_diag_other", DURATION_POST_DIAG_OTHER
        )
        self.dur_review = kwargs.get("dur_review", DURATION_REVIEW)

        # Capacity slots
        self.staff_screening = kwargs.get("staff_screening", SCREENING_SLOT)
        self.staff_pre_assessment = kwargs.get(
            "staff_pre_assessment", PRE_ASSESSMENT_SLOT
        )
        self.staff_assessment = kwargs.get(
            "staff_assessment", ASSESSMENT_SLOT
        )
        self.staff_further_assessment = kwargs.get(
            "staff_further_assessment", FURTHER_ASSESSMENT_SLOT
        )
        self.staff_post_diag_clinical = kwargs.get(
            "staff_post_diag_clinical", POST_DIAG_CLINICAL_SLOT
        )
        self.staff_post_diag_other = kwargs.get(
            "staff_post_diag_other", POST_DIAG_OTHER_SLOT
        )
        self.staff_review = kwargs.get("staff_review", REVIEW_SLOT)

        self.init_results_variables()
        self.init_sampling()

    def set_random_no_set(self, rep):
        """
        Set the replication seed and rebuild random streams.

        Parameters
        ----------
        random_number_set : int
            Seed value for the next simulation run.

        Returns
        -------
        None
            Distribution objects are reinitialised in place.
        """
        self.random_number_set = self.base_random_number_set + rep
        self.init_sampling()

    def init_sampling(self):
        """
        Create random distributions for arrivals and pathway events.

        Returns
        -------
        None
            Distribution objects are attached to the experiment instance.
        """
     
        # Array to store raw integer seeds for distribution consumption
        int_seeds = [None] * self.n_streams

        if self.use_fixed_seed:
            # Step 1: Create a master sequence for this specific replication run
            master_seq = np.random.SeedSequence(self.random_number_set)
            
            # Step 2: Spawn N separate, non-overlapping child sequences
            child_sequences = master_seq.spawn(self.n_streams)
            
            # Step 3: Extract a clean 32-bit integer from each child's entropy pool
            for i in range(self.n_streams):
                int_seeds[i] = int(child_sequences[i].generate_state(1)[0])    

        def to_days(hours):
            """Convert a list of hour durations to days.

            Parameters
            ----------
            hours : sequence of float
                Duration values measured in hours.

            Returns
            -------
            list of float
                Duration values converted to days.
            """
            return [h / 24.0 for h in hours]

        self.iat_dist = Exponential(
            self.iat,
            random_seed=int_seeds[0]
            )
        
        self.screening_time_dist = Triangular(
            *to_days(self.dur_screening),
            random_seed=int_seeds[1]
                if self.use_fixed_seed
                else None
            )
        
        self.pre_ass_time_dist = Triangular(
            *to_days(self.dur_pre_assessment),
            random_seed=int_seeds[2]
            )
        
        self.assessment_time_dist = Triangular(
            *to_days(self.dur_assessment),
            random_seed=int_seeds[3]
        )
        self.referral_reject_dist = Bernoulli(
            self.triage_rejected,
            random_seed=int_seeds[4]
        )
        self.screening_discharge_dist = Bernoulli(
            self.screening_discharge,
            random_seed=int_seeds[5]
        )
        self.pre_ass_reject_dist = Bernoulli(
            self.pre_assessment_rejection,
            random_seed=int_seeds[6]
        )
        self.non_diag_at_assessment_dist = Bernoulli(
            self.pct_non_diag_at_assessment,
            random_seed=int_seeds[7]
        )
        self.further_assessment_time_dist = Triangular(
            *to_days(self.dur_further_assessment),
            random_seed=int_seeds[8]
        )
        self.non_diag_at_further_assessment_dist = Bernoulli(
            self.pct_non_diag_at_further_assessment,
            random_seed=int_seeds[9]
        )
        self.post_diag_clinical_dist = Bernoulli(
            self.pct_post_diag_clinical,
            random_seed=int_seeds[10]
        )
        self.post_diag_clinical_time_dist = Triangular(
            *to_days(self.dur_post_diag_clinical),
            random_seed=int_seeds[11]
        )
        self.post_diag_other_time_dist = Triangular(
            *to_days(self.dur_post_diag_other),
            random_seed=int_seeds[12]
        )
        self.review_time_dist = Triangular(
            *to_days(self.dur_review),
            random_seed=int_seeds[13]
        )
        self.removal_discharge_dist = Bernoulli(
            self.pct_removal_discharge,
            random_seed=int_seeds[14]
        )

    def init_results_variables(self):
        """
        Initialise all pathway flow counters to zero.

        Returns
        -------
        None
            The ``results`` dictionary is replaced in place.
        """
        self.results = {
            "ARRIVED_TOTAL": 0,
            "EXIT_TOTAL": 0,
            "CLINICAL_COMPLETED_TOTAL": 0,
            "ARRIVED_REFERRAL": 0,
            "FLOW_REFERRAL_ACCEPTED": 0,
            "EXIT_REFERRAL_REJECTED": 0,
            "ARRIVED_SCREENING": 0,
            "SERVICE_SCREENING_COMPLETED": 0,
            "FLOW_SCREENING_PASSED": 0,
            "EXIT_SCREENING_DISCHARGED": 0,
            "ARRIVED_PRE_ASSESS": 0,
            "SERVICE_PRE_ASSESS_COMPLETED": 0,
            "FLOW_PRE_ASSESS_PASSED": 0,
            "EXIT_PRE_ASSESS_REJECTED": 0,
            "ARRIVED_ASSESSMENT": 0,
            "SERVICE_ASSESSMENT_COMPLETED": 0,
            "FLOW_ASSESSMENT_PASSED": 0,
            "EXIT_ASSESSMENT_NON_DIAGNOSIS": 0,
            "ARRIVED_FURTHER_ASSESS": 0,
            "SERVICE_FURTHER_ASSESS_COMPLETED": 0,
            "FLOW_FURTHER_ASSESS_PASSED": 0,
            "EXIT_FURTHER_NON_DIAGNOSIS": 0,
            "FLOW_DIAGNOSIS_CONFIRMED": 0,
            "FLOW_POST_DIAG_CLINICAL_ACCEPTED": 0,
            "SERVICE_POST_DIAG_CLINICAL_COMPLETED": 0,
            "FLOW_POST_DIAG_OTHER_ACCEPTED": 0,
            "SERVICE_POST_DIAG_OTHER_COMPLETED": 0,
            "EXIT_FORMAL_DISCHARGE": 0,
            "EXIT_SELF_REMOVED": 0,
        }





## 6. Patient Pathway Logic

The `Patient` class describes the journey of one referral through the service. Each patient enters the pathway, faces probabilistic decisions at clinical gates, waits for appointment slots, receives service time, and then either progresses or exits.

The code uses SimPy generator logic: `yield` pauses the patient until a resource slot is granted or a service duration has passed. This is what allows many patients to move through the system at the same simulated time while competing for limited capacity.


In [74]:
# ============================================================
# PATIENT LIFECYCLE WITH ENTER/EXIT TRACKING PER STAGE
# ============================================================
class Patient:
    """
    Represent one patient moving through the simulated pathway.
    Tracks and prints simulation times and days for every step entry.
    """

    def __init__(self, patient_id, system):
        """Initialise patient state and connect system references."""
        self.patient_id = patient_id
        self.system = system
        self.env = system.env
        self.args = system.args
        self.auditor = system.auditor
        
        # Reference list to translate simulation days into text strings
        self.DAYS_OF_WEEK = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

    def _get_timestamp(self):
        """Helper to return current time and formatted weekday string."""
        day_name = self.DAYS_OF_WEEK[int(self.env.now % 7)]
        return f"[Time {self.env.now:.3f} | {day_name}]"

    def process(self):
        """
        Execute the full patient lifecycle, printing structural timestamps
        at every queue and stage entry point.
        """

        def patient_trace(message):
            """Print a patient-scoped trace message."""
            trace(f"{self._get_timestamp()} Patient_{self.patient_id}: {message}")

        referral_start_time = self.env.now
        self.args["ARRIVED_TOTAL"] += 1
        self.args["ARRIVED_REFERRAL"] += 1

        trace(f"{self._get_timestamp()} Patient {self.patient_id} entered system. Referral submitted.")

        # ============================================================
        # 1. REFERRAL TRIAGE GATEWAY NODE (Instantaneous Check)
        # ============================================================
        if self.system.referral_reject_dist.sample() == 1:
            self.args["EXIT_REFERRAL_REJECTED"] += 1
            self.args["EXIT_TOTAL"] += 1
            trace(f"{self._get_timestamp()} Patient {self.patient_id} Exit - Referral Rejected at Triage.")
            return
        self.args["FLOW_REFERRAL_ACCEPTED"] += 1


        # ============================================================
        # 2. CLINICAL TRIAGE SCREENING QUEUE AND PROCESS
        # ============================================================
        self.args["ARRIVED_SCREENING"] += 1
        screening_queue = self.system.screening_resource.count_queue()
        self.auditor.record_queue_length("screening", screening_queue)
        trace(f"{self._get_timestamp()} Patient {self.patient_id} queued for SCREENING (Queue pos: {screening_queue}).")

        yield self.system.screening_resource.request()
        
        # --- STAGE ENTRY POINT ---
        trace(f"{self._get_timestamp()} >> Patient {self.patient_id} officially ENTERED SCREENING stage.")
        yield self.env.timeout(self.system.screening_time_dist.sample())
        self.args["SERVICE_SCREENING_COMPLETED"] += 1
        self.auditor.record_rtt("screening", self.env.now - referral_start_time)

        if self.system.screening_discharge_dist.sample() == 1:
            self.args["EXIT_SCREENING_DISCHARGED"] += 1
            self.args["EXIT_TOTAL"] += 1
            trace(f"{self._get_timestamp()} Patient {self.patient_id} Exit - Discharged after Screening.")
            return
        self.args["FLOW_SCREENING_PASSED"] += 1


        # ============================================================
        # 3. CLINICAL PRE-ASSESSMENT QUEUE AND PROCESS
        # ============================================================
        self.args["ARRIVED_PRE_ASSESS"] += 1
        pre_assessment_queue = self.system.pre_assessment_resource.count_queue()
        self.auditor.record_queue_length("pre_assessment", pre_assessment_queue)
        trace(f"{self._get_timestamp()} Patient {self.patient_id} queued for PRE-ASSESSMENT (Queue pos: {pre_assessment_queue}).")

        yield self.system.pre_assessment_resource.request()
        
        # --- STAGE ENTRY POINT ---
        trace(f"{self._get_timestamp()} >> Patient {self.patient_id} officially ENTERED PRE-ASSESSMENT stage.")
        yield self.env.timeout(self.system.pre_ass_time_dist.sample())
        self.args["SERVICE_PRE_ASSESS_COMPLETED"] += 1
        self.auditor.record_rtt("pre_assessment", self.env.now - referral_start_time)

        if self.system.pre_ass_reject_dist.sample() == 1:
            self.args["EXIT_PRE_ASSESS_REJECTED"] += 1
            self.args["EXIT_TOTAL"] += 1
            trace(f"{self._get_timestamp()} Patient {self.patient_id} Exit - Rejected at Pre-Assessment stage.")
            return
        self.args["FLOW_PRE_ASSESS_PASSED"] += 1


        # ============================================================
        # 4. CORE AUTISM CLINICAL ASSESSMENT QUEUE AND PROCESS
        # ============================================================
        self.args["ARRIVED_ASSESSMENT"] += 1
        assessment_queue = self.system.assessment_resource.count_queue()
        self.auditor.record_queue_length("assessment", assessment_queue)
        trace(f"{self._get_timestamp()} Patient {self.patient_id} queued for CORE ASSESSMENT (Queue pos: {assessment_queue}).")

        yield self.system.assessment_resource.request()
        
        # --- STAGE ENTRY POINT ---
        trace(f"{self._get_timestamp()} >> Patient {self.patient_id} officially ENTERED CORE ASSESSMENT stage.")
        yield self.env.timeout(self.system.assessment_time_dist.sample())
        self.args["SERVICE_ASSESSMENT_COMPLETED"] += 1
        self.auditor.record_rtt("assessment", self.env.now - referral_start_time)

        if self.system.non_diag_at_assessment_dist.sample() == 1:
            self.args["EXIT_ASSESSMENT_NON_DIAGNOSIS"] += 1
            self.args["EXIT_TOTAL"] += 1
            trace(f"{self._get_timestamp()} Patient {self.patient_id} Exit - Non-Diagnosis at Core Assessment.")
            return
        self.args["FLOW_ASSESSMENT_PASSED"] += 1


        # ============================================================
        # 5. MULTIDISCIPLINARY SECONDARY FURTHER ASSESSMENT QUEUE
        # ============================================================
        self.args["ARRIVED_FURTHER_ASSESS"] += 1
        further_queue = self.system.further_assessment_resource.count_queue()
        self.auditor.record_queue_length("further_assessment", further_queue)
        trace(f"{self._get_timestamp()} Patient {self.patient_id} queued for FURTHER ASSESSMENT (Queue pos: {further_queue}).")

        yield self.system.further_assessment_resource.request()
        
        # --- STAGE ENTRY POINT ---
        trace(f"{self._get_timestamp()} >> Patient {self.patient_id} officially ENTERED FURTHER ASSESSMENT stage.")
        yield self.env.timeout(self.system.further_assessment_time_dist.sample())
        self.args["SERVICE_FURTHER_ASSESS_COMPLETED"] += 1
        self.auditor.record_rtt("further_assessment", self.env.now - referral_start_time)

        if self.system.non_diag_at_further_assessment_dist.sample() == 1:
            self.args["EXIT_FURTHER_NON_DIAGNOSIS"] += 1
            self.args["EXIT_TOTAL"] += 1
            trace(f"{self._get_timestamp()} Patient {self.patient_id} Exit - Non-Diagnosis at Further Assessment.")
            return

        self.args["FLOW_FURTHER_ASSESS_PASSED"] += 1
        self.args["FLOW_DIAGNOSIS_CONFIRMED"] += 1
        self.auditor.record_rtt("diagnosis", self.env.now - referral_start_time)


        # ============================================================
        # 6. POST-DIAGNOSTIC SUPPORT OPTIONS NODE SELECTION
        # ============================================================
        if self.system.post_diag_clinical_dist.sample() == 1:
            self.args["FLOW_POST_DIAG_CLINICAL_ACCEPTED"] += 1
            post_clinical_queue = self.system.post_diag_clinical_resource.count_queue()
            self.auditor.record_queue_length("post_diag_clinical", post_clinical_queue)
            trace(f"{self._get_timestamp()} Patient {self.patient_id} queued for POST-DIAG CLINICAL SUPPORT.")
            
            yield self.system.post_diag_clinical_resource.request()
            
            # --- STAGE ENTRY POINT ---
            trace(f"{self._get_timestamp()} >> Patient {self.patient_id} officially ENTERED POST-DIAG CLINICAL SUPPORT stage.")
            yield self.env.timeout(self.system.post_diag_clinical_time_dist.sample())
            self.args["SERVICE_POST_DIAG_CLINICAL_COMPLETED"] += 1
        else:
            self.args["FLOW_POST_DIAG_OTHER_ACCEPTED"] += 1
            post_other_queue = self.system.post_diag_other_resource.count_queue()
            self.auditor.record_queue_length("post_diag_other", post_other_queue)
            trace(f"{self._get_timestamp()} Patient {self.patient_id} queued for POST-DIAG OTHER SUPPORT.")
            
            yield self.system.post_diag_other_resource.request()
            
            # --- STAGE ENTRY POINT ---
            trace(f"{self._get_timestamp()} >> Patient {self.patient_id} officially ENTERED POST-DIAG OTHER SUPPORT stage.")
            yield self.env.timeout(self.system.post_diag_other_time_dist.sample())
            self.args["SERVICE_POST_DIAG_OTHER_COMPLETED"] += 1


        # ============================================================
        # 7. FINAL CASE DISCHARGE REVIEW PROCESS
        # ============================================================
        review_queue = self.system.review_resource.count_queue()
        self.auditor.record_queue_length("review", review_queue)
        trace(f"{self._get_timestamp()} Patient {self.patient_id} queued for FINAL CASE DISCHARGE REVIEW.")
        
        yield self.system.review_resource.request()
        
        # --- STAGE ENTRY POINT ---
        trace(f"{self._get_timestamp()} >> Patient {self.patient_id} officially ENTERED FINAL DISCHARGE REVIEW stage.")
        yield self.env.timeout(self.system.review_time_dist.sample())

        if self.system.removal_discharge_dist.sample() == 1:
            self.args["EXIT_FORMAL_DISCHARGE"] += 1
            trace(f"{self._get_timestamp()} Patient {self.patient_id} Complete - Formal Discharge.")
        else:
            self.args["EXIT_SELF_REMOVED"] += 1
            trace(f"{self._get_timestamp()} Patient {self.patient_id} Complete - Self-Removal path.")

        self.args["CLINICAL_COMPLETED_TOTAL"] += 1
        self.args["EXIT_TOTAL"] += 1

## 7. Calendar-Aware Resource Setup

This section defines a custom appointment-booking resource used throughout the simulation. Patients requesting an appointment join a first-in, first-out (FIFO) queue and wait until a slot becomes available.

Each simulated day, the resource checks the current weekday and releases the number of appointment slots specified for that day. Available slots are immediately assigned to the patients who have been waiting the longest in the queue. If more patients are waiting than slots are available, the remaining patients stay in the queue until future capacity is released. If fewer patients are waiting than available slots, the unused slots are recorded.

This approach allows the model to represent appointment-based services where capacity is released according to a weekly schedule rather than being continuously available.

In [75]:
# ============================================================
# CALENDAR-AWARE QUEUE RESOURCE WITH WEEKDAY CAPACITY RELEASE
# ============================================================
class CalendarAwareQueueResource:
    """
    Release appointment slots by weekday and manage a FIFO queue.

    Parameters
    ----------
    env : simpy.Environment
        Simulation environment used to create events and timeouts.
    weekly_slots : dict
        Mapping from weekday index to the number of slots released.
    name : str, default "resource"
        Human-readable resource name used in summaries.
    """

    def __init__(self, env, weekly_slots, name="resource"):
        """
        Initialise queue state and start the slot scheduler.

        Parameters
        ----------
        env : simpy.Environment
            Simulation environment used by the resource.
        weekly_slots : dict
            Weekday-to-slot mapping for capacity release.
        name : str, default "resource"
            Name recorded in resource summaries.

        Returns
        -------
        None
            The scheduler process is registered with the environment.
        """
        self.env = env
        self.name = name
        self.weekly_slots = weekly_slots
        self.queue = deque()

        # Performance Counters
        self.released_slots = 0
        self.used_slots = 0
        self.unused_slots = 0
        self.max_queue_length = 0

        self.env.process(self._slot_scheduler())

    def request(self):
        """
        Request the next available appointment slot.

        Returns
        -------
        simpy.events.Event
            Event that succeeds when a released slot is assigned.
        """
        evt = self.env.event()
        self.queue.append(evt)
        self.max_queue_length = max(self.max_queue_length, len(self.queue))
        return evt

    def _slot_scheduler(self):
        """
        Release slots each simulated day and serve queued requests.

        Yields
        ------
        simpy.events.Timeout
            One-day timeout between capacity release cycles.
        """
        while True:
            today = int(math.floor(self.env.now + 0.00001))
            weekday = today % 7
            slots_today = self.weekly_slots.get(weekday, 0)
            self.released_slots += slots_today

            while slots_today > 0 and self.queue:
                patient_event = self.queue.popleft()
                if not patient_event.triggered:
                    patient_event.succeed()
                    self.used_slots += 1
                    slots_today -= 1

            self.unused_slots += slots_today
            yield self.env.timeout(1.0)

    def count_queue(self):
        """
        Return the current number of queued patients.

        Returns
        -------
        int
            Current FIFO queue length.
        """
        return len(self.queue)

    @property
    def utilisation(self):
        """
        Return the fraction of released slots that were used.

        Returns
        -------
        float
            Slot utilisation bounded between 0 and 1.
        """
        if self.released_slots <= 0:
            return 0.0
        return min(self.used_slots / self.released_slots, 1.0)

    def get_summary(self):
        """
        Summarise current queue and capacity statistics.

        Returns
        -------
        dict
            Resource backlog, slot counts, and utilisation.
        """
        return {
            "resource_name": self.name,
            "queue_length": self.count_queue(),
            "queue_backlog": self.count_queue(),
            "max_queue_length": self.max_queue_length,
            "released_slots": self.released_slots,
            "used_slots": self.used_slots,
            "unused_slots": self.unused_slots,
            "utilisation_pct": round(self.utilisation * 100, 2),
        }


## 8. System Orchestration

The `AutismPathwaySystem` connects the experiment settings, resources, random distributions, and patient generation process. It creates one resource for each pathway stage and starts referrals according to the weekday arrival process.

This class acts as the model wiring layer: the patient class defines individual behaviour, while this system class defines the environment those patients move through.


In [76]:
# ============================================================
# HEALTH PATHWAY ORCHESTRATION SYSTEM CORE
# ============================================================
class AutismPathwaySystem:
    """
    Coordinate resources, distributions, and patient arrivals.

    Parameters
    ----------
    env : simpy.Environment
        Simulation environment for the run.
    experiment : Experiment
        Parameter container and random distribution source.
    arrival_stop : float, default RUN_LENGTH
        Final simulated time at which new referrals stop arriving.
    """

    def __init__(self, env, experiment, arrival_stop=RUN_LENGTH):
        """
        Initialise pathway resources and distribution aliases.

        Parameters
        ----------
        env : simpy.Environment
            Simulation environment for the run.
        experiment : Experiment
            Experiment containing parameters, counters, and distributions.
        arrival_stop : float, default RUN_LENGTH
            Time in days after which arrivals stop.

        Returns
        -------
        None
            System state and resources are attached to the object.
        """
        self.env = env
        self.experiment = experiment
        self.args = experiment.results
        self.auditor = experiment.auditor
        self.arrival_stop = arrival_stop

        # Link stochastic distribution samples
        self.referral_reject_dist = experiment.referral_reject_dist
        self.screening_time_dist = experiment.screening_time_dist
        self.screening_discharge_dist = experiment.screening_discharge_dist
        self.pre_ass_time_dist = experiment.pre_ass_time_dist
        self.pre_ass_reject_dist = experiment.pre_ass_reject_dist
        self.assessment_time_dist = experiment.assessment_time_dist
        self.non_diag_at_assessment_dist = (
            experiment.non_diag_at_assessment_dist
        )
        self.further_assessment_time_dist = (
            experiment.further_assessment_time_dist
        )
        self.non_diag_at_further_assessment_dist = (
            experiment.non_diag_at_further_assessment_dist
        )
        self.post_diag_clinical_dist = experiment.post_diag_clinical_dist
        self.post_diag_clinical_time_dist = (
            experiment.post_diag_clinical_time_dist
        )
        self.post_diag_other_time_dist = experiment.post_diag_other_time_dist
        self.review_time_dist = experiment.review_time_dist
        self.removal_discharge_dist = experiment.removal_discharge_dist

        # Map Monday (0) through Friday (4) to the same slot capacity template
        weekdays_template = lambda slots: {i: slots for i in range(5)}

        # Resource dictionary used by evaluation loops.
        self.resources = {
            "screening": CalendarAwareQueueResource(
                env,
                weekdays_template(experiment.staff_screening),
                "screening",
            ),
            "pre_assessment": CalendarAwareQueueResource(
                env,
                weekdays_template(experiment.staff_pre_assessment),
                "pre_assessment",
            ),
            "assessment": CalendarAwareQueueResource(
                env,
                weekdays_template(experiment.staff_assessment),
                "assessment",
            ),
            "further_assessment": CalendarAwareQueueResource(
                env,
                weekdays_template(experiment.staff_further_assessment),
                "further_assessment",
            ),
            "post_diag_clinical": CalendarAwareQueueResource(
                env,
                weekdays_template(experiment.staff_post_diag_clinical),
                "post_diag_clinical",
            ),
            "post_diag_other": CalendarAwareQueueResource(
                env,
                weekdays_template(experiment.staff_post_diag_other),
                "post_diag_other",
            ),
            "review": CalendarAwareQueueResource(
                env, weekdays_template(experiment.staff_review), "review"
            ),
        }

        # Exposed aliases for clean code mapping inside Patient process paths
        self.screening_resource = self.resources["screening"]
        self.pre_assessment_resource = self.resources["pre_assessment"]
        self.assessment_resource = self.resources["assessment"]
        self.further_assessment_resource = self.resources[
            "further_assessment"
        ]
        self.post_diag_clinical_resource = self.resources[
            "post_diag_clinical"
        ]
        self.post_diag_other_resource = self.resources["post_diag_other"]
        self.review_resource = self.resources["review"]

    def run(self):
        """
        Generate weekday referral arrivals until the stop time.
        Logs weekend events and skips Saturday/Sunday safely.

        Yields
        ------
        simpy.events.Timeout
            Inter-arrival and weekend-delay timeouts.
        """

        self.env.process(
            self._audit_loop()
        )  # Start background data logging loop

        for pid in itertools.count(start=1):
            if self.env.now >= self.arrival_stop:
                break

            delay = self.experiment.iat_dist.sample()
            yield self.env.timeout(delay)

            # THE WEEKEND GATEKEEPER
            while True:
                if self.env.now >= self.arrival_stop:
                    break
                
                current_week_time = self.env.now % 7

                # If simulation time lands on or past Saturday 00:00 (5.0)
                if current_week_time >= 5.0:
                    
                    # Calculate remaining time until Monday morning
                    time_to_monday = 7.0 - current_week_time
                    yield self.env.timeout(time_to_monday)
            
                    # Sample a clean, new arrival delay starting from Monday 00:00
                    delay = self.experiment.iat_dist.sample()
                    yield self.env.timeout(delay)
                else:
                    break

            # SAFETY CHECK & EVENT TRIGGER
            if self.env.now < self.arrival_stop:
                day_index = int(self.env.now % 7)

                patient = Patient(pid, self)
                self.env.process(patient.process())
                
    def _audit_loop(self):
        """
        Capture resource statistics once per simulated day.

        Yields
        ------
        simpy.events.Timeout
            Daily timeout between audit snapshots.
        """
        while True:
            yield self.env.timeout(1.0)
            for name, res in self.resources.items():
                self.auditor.capture_resource_stats(name, res)


## 9. Single Run Function

The `single_run` function executes one full simulation replication. It resets counters, applies the random seed for the selected replication, creates the SimPy environment, runs the pathway, and then returns a dictionary of KPI results.

This wrapper is important because it gives later testing and replication sections one consistent way to run the model.


In [77]:
def single_run(experiment, rep=0, run_length=1825):
    """
    Run one complete simulation replication.

    Parameters
    ----------
    experiment : Experiment
        Configured experiment object for the replication.
    rep : int, default 0
        Replication index used as the deterministic seed value.
    run_length : float, default 1825
        Simulation horizon in days.

    Returns
    -------
    dict
        Flow counters and summary KPIs for the replication.
    """
    experiment.auditor.reset()

    experiment.init_results_variables()
    
    
    # Set the random seed for this replication if deterministic sampling is enabled.
    if experiment.use_fixed_seed:
        experiment.set_random_no_set(rep)

    env = simpy.Environment()

    system = AutismPathwaySystem(env, experiment, arrival_stop=run_length)

    env.process(system.run())

    env.run(until=run_length)

    # Capture resource statistics ONCE
    for stage_name, resource in system.resources.items():

        experiment.auditor.capture_resource_stats(stage_name, resource)

    experiment.results["IN_SYSTEM_END"] = (
        experiment.results["ARRIVED_TOTAL"] - experiment.results["EXIT_TOTAL"]
    )

    experiment.results["SYSTEM_COMPLETED_TOTAL"] = experiment.results[
        "EXIT_TOTAL"
    ]

    results = experiment.results.copy()
    results['SEED_USED'] = experiment.random_number_set 

    summary_metrics = experiment.auditor.summarize(results, run_length)

    results.update(summary_metrics)

    return results


## 10. Quick Trace Test Run

This short run turns tracing on for three simulated days. The purpose is not to estimate final performance, but to inspect whether patients are entering, queueing, progressing, and exiting in the expected order.

A trace run is a practical early verification step because it exposes event sequencing errors that may be hidden in aggregate results.


In [78]:
TRACE = True

test_days = 3

test_experiment = Experiment(auditor=Audit())

test_results = single_run(test_experiment, rep=0, run_length=test_days)


[Time 0.221 | Monday] Patient 1 entered system. Referral submitted.
[Time 0.221 | Monday] Patient 1 Exit - Referral Rejected at Triage.
[Time 0.360 | Monday] Patient 2 entered system. Referral submitted.
[Time 0.360 | Monday] Patient 2 Exit - Referral Rejected at Triage.
[Time 0.499 | Monday] Patient 3 entered system. Referral submitted.
[Time 0.499 | Monday] Patient 3 queued for SCREENING (Queue pos: 0).
[Time 1.000 | Tuesday] >> Patient 3 officially ENTERED SCREENING stage.
[Time 1.032 | Tuesday] Patient 3 queued for PRE-ASSESSMENT (Queue pos: 0).
[Time 1.240 | Tuesday] Patient 4 entered system. Referral submitted.
[Time 1.240 | Tuesday] Patient 4 queued for SCREENING (Queue pos: 0).
[Time 1.257 | Tuesday] Patient 5 entered system. Referral submitted.
[Time 1.257 | Tuesday] Patient 5 Exit - Referral Rejected at Triage.
[Time 1.487 | Tuesday] Patient 6 entered system. Referral submitted.
[Time 1.487 | Tuesday] Patient 6 queued for SCREENING (Queue pos: 1).
[Time 1.770 | Tuesday] Patie

## 11. Main Results Run

This cell runs the baseline model for the full five-year horizon and prints headline KPIs. The outputs include arrivals, exits, clinical completions, pathway exits by reason, referral-to-treatment waiting times, backlog, slot use, and overall utilisation.

These figures provide the main operational view of the current pathway assumptions.


In [79]:
TRACE = False

experiment = Experiment(auditor=Audit())

results = single_run(experiment, rep=0, run_length=RUN_LENGTH)

print("\nResults:")

print(
    f"refferals per day: {REFERRALS_PER_DAY}"
    f":RUN LENGTH: {RUN_LENGTH} days "
    f"({RUN_LENGTH / 365:.2f} YEARS)"
)

print(
    f"SLOTS PER WEEK:\n"
    f" screening: {SCREENING_SLOT}\n"
    f" pre-assessment: {PRE_ASSESSMENT_SLOT}\n"
    f" assessment: {ASSESSMENT_SLOT}\n"
    f" further assessment: {FURTHER_ASSESSMENT_SLOT}\n"
    f" post-diag clinical: {POST_DIAG_CLINICAL_SLOT}\n"
    f" post-diag other: {POST_DIAG_OTHER_SLOT}\n"
    f" review: {REVIEW_SLOT}"
)

print(f"Total Arrivals: {results['ARRIVED_TOTAL']}")
print(f"Total Exits: {results['EXIT_TOTAL']}")
print(f"Total In System at End: {results['IN_SYSTEM_END']}")

total_completed = results["CLINICAL_COMPLETED_TOTAL"]
print(f"Total Clinically Completed: {total_completed}")

print(
    f"Referral Rejections at Triage: " f"{results['EXIT_REFERRAL_REJECTED']}"
)

print(f"Screening Discharges: " f"{results['EXIT_SCREENING_DISCHARGED']}")

print(f"Pre-Assessment Rejections: " f"{results['EXIT_PRE_ASSESS_REJECTED']}")

print(
    f"Non-Diagnosis at Assessment: "
    f"{results['EXIT_ASSESSMENT_NON_DIAGNOSIS']}"
)

print(
    f"Non-Diagnosis at Further Assessment: "
    f"{results['EXIT_FURTHER_NON_DIAGNOSIS']}"
)

print(f"Formal Discharges: " f"{results['EXIT_FORMAL_DISCHARGE']}")

print(f"Self-Removals: " f"{results['EXIT_SELF_REMOVED']}")

print(
    f"Referral to Screening RTT (days): "
    f"{results['ACCESS_REFERRAL_TO_SCREENING_RTT_DAYS']:.2f}"
)

print(
    f"Referral to Pre-Assessment RTT (days): "
    f"{results['ACCESS_REFERRAL_TO_PRE_ASSESSMENT_RTT_DAYS']:.2f}"
)

print(
    f"Referral to Assessment RTT (days): "
    f"{results['ACCESS_REFERRAL_TO_ASSESSMENT_RTT_DAYS']:.2f}"
)

print(
    f"Referral to Further Assessment RTT (days): "
    f"{results['ACCESS_REFERRAL_TO_FURTHER_ASSESSMENT_RTT_DAYS']:.2f}"
)

print(
    f"Referral to Diagnosis RTT (days): "
    f"{results['ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS']:.2f}"
)

total_backlog = results["QUEUE_TOTAL_BACKLOG"]
print(f"Total System Backlog at End: {total_backlog:.0f}")

# DEBUG VALUES
print(f"Total Slots Used: " f"{results['CAPACITY_TOTAL_SLOT_USAGE']:.0f}")

print(
    f"TTotal Slots Released: "
    f"{results['CAPACITY_TOTAL_SLOT_RELEASED']:.0f}"
)


# CORRECT UTILISATION OUTPUT
print(
    f"Overall Capacity Utilisation: "
    f"{results['OVERALL_SYSTEM_UTILISATION']:.2f}%"
)

print(
    f"Diagnosis Rate among Arrivals: "
    f"{results['FLOW_DIAGNOSIS_RATE_PCT']:.2f}%"
)



Results:
refferals per day: 5:RUN LENGTH: 1825 days (5.00 YEARS)
SLOTS PER WEEK:
 screening: 7
 pre-assessment: 3
 assessment: 5
 further assessment: 3
 post-diag clinical: 4
 post-diag other: 2
 review: 2
Total Arrivals: 6609
Total Exits: 5951
Total In System at End: 658
Total Clinically Completed: 2595
Referral Rejections at Triage: 1651
Screening Discharges: 596
Pre-Assessment Rejections: 290
Non-Diagnosis at Assessment: 520
Non-Diagnosis at Further Assessment: 299
Formal Discharges: 2289
Self-Removals: 306
Referral to Screening RTT (days): 0.95
Referral to Pre-Assessment RTT (days): 88.73
Referral to Assessment RTT (days): 90.16
Referral to Further Assessment RTT (days): 91.21
Referral to Diagnosis RTT (days): 91.34
Total System Backlog at End: 658
Total Slots Used: 20923
TTotal Slots Released: 33930
Overall Capacity Utilisation: 61.67%
Diagnosis Rate among Arrivals: 42.14%


## 12. 100% Triage Rejection Test

This is a simple boundary-condition test. By setting the triage rejection probability to `1.0`, every referral should be rejected immediately and no patient should reach screening or complete the clinical pathway.

The assertions check that the counters match this expected behaviour. If any assertion fails, the referral gateway logic or result counting is inconsistent.


In [80]:
# TESTING: increase referral rejection to 100%
TRACE = False

# FIX: Corrected spelling from 'traige_rejected' to 'triage_rejected'
experiment = Experiment(auditor=Audit(), triage_rejected=1.0)
test_results = single_run(experiment, rep=0, run_length=RUN_LENGTH)

print("=== 100% Triage Rejection Test ===")
print(f"ARRIVED_TOTAL                  : {test_results['ARRIVED_TOTAL']}")
print(f"ARRIVED_REFERRAL               : {test_results['ARRIVED_REFERRAL']}")
print(
    "FLOW_REFERRAL_ACCEPTED         : "
    f"{test_results['FLOW_REFERRAL_ACCEPTED']}"
)
print(
    "EXIT_REFERRAL_REJECTED         : "
    f"{test_results['EXIT_REFERRAL_REJECTED']}"
)
print(f"ARRIVED_SCREENING              : {test_results['ARRIVED_SCREENING']}")
print(
    "FLOW_SCREENING_PASSED          : "
    f"{test_results['FLOW_SCREENING_PASSED']}"
)
print(
    "CLINICAL_COMPLETED_TOTAL       : "
    f"{test_results['CLINICAL_COMPLETED_TOTAL']}"
)
print()

# Operational assertions to validate the boundary conditions
assert (
    test_results["EXIT_REFERRAL_REJECTED"] == test_results["ARRIVED_REFERRAL"]
), "FAIL: not all referrals were rejected"
assert (
    test_results["FLOW_REFERRAL_ACCEPTED"] == 0
), "FAIL: some referrals were accepted past triage"
assert (
    test_results["ARRIVED_SCREENING"] == 0
), "FAIL: patients reached screening"
assert (
    test_results["FLOW_SCREENING_PASSED"] == 0
), "FAIL: patients passed screening"
assert (
    test_results["CLINICAL_COMPLETED_TOTAL"] == 0
), "FAIL: patients completed the clinical journey"

print("All assertions passed: 100% triage rejection behaves correctly.")


=== 100% Triage Rejection Test ===
ARRIVED_TOTAL                  : 6609
ARRIVED_REFERRAL               : 6609
FLOW_REFERRAL_ACCEPTED         : 0
EXIT_REFERRAL_REJECTED         : 6609
ARRIVED_SCREENING              : 0
FLOW_SCREENING_PASSED          : 0
CLINICAL_COMPLETED_TOTAL       : 0

All assertions passed: 100% triage rejection behaves correctly.


## 13. Stage Boundary Tests: 100% Exits and 0% Exits

This section checks that pathway branching works correctly at each major decision point.

### Why This Matters

The model uses probabilities to decide whether a patient exits or moves forward. Boundary tests force these probabilities to their extremes so that the expected answer is unambiguous.

### What Is Tested

The tested stages are triage, screening, pre-assessment, assessment, and further assessment. For each stage, the suite checks two scenarios:

1. `1.0`, meaning every patient who reaches that decision should exit there.
2. `0.0`, meaning no patient who reaches that decision should exit there.

### Expected Behaviour

For a 100% exit setting, exits at that stage should equal the number of patients who reached the decision point, pass-through should be zero, and downstream entry should be zero.

For a 0% exit setting, exits at that stage should be zero, pass-through should match the relevant stage count, and at least some patients should reach the next stage.

These checks map directly to the transition logic in `Patient.process()` and the counters stored in `Experiment.results`.


In [81]:
# TESTING: stage boundary tests (100% exits and 0% exits)
TRACE = False

stage_configs = {
    "triage": {
        "param": "triage_rejected",  # FIX: Corrected typo spelling
        "arrived_key": "ARRIVED_REFERRAL",
        "exit_key": "EXIT_REFERRAL_REJECTED",
        "pass_key": "FLOW_REFERRAL_ACCEPTED",
        "downstream_entry_key": "ARRIVED_SCREENING",
    },
    "screening": {
        "param": "screening_discharge",
        "arrived_key": "ARRIVED_SCREENING",
        "exit_key": "EXIT_SCREENING_DISCHARGED",
        "pass_key": "FLOW_SCREENING_PASSED",
        "downstream_entry_key": "ARRIVED_PRE_ASSESS",
        "service_key": "SERVICE_SCREENING_COMPLETED",
    },
    "pre_assessment": {
        "param": "pre_assessment_rejection",
        "arrived_key": "ARRIVED_PRE_ASSESS",
        "exit_key": "EXIT_PRE_ASSESS_REJECTED",
        "pass_key": "FLOW_PRE_ASSESS_PASSED",
        "downstream_entry_key": "ARRIVED_ASSESSMENT",
        "service_key": "SERVICE_PRE_ASSESS_COMPLETED",
    },
    "assessment": {
        "param": "pct_non_diag_at_assessment",
        "arrived_key": "ARRIVED_ASSESSMENT",
        "exit_key": "EXIT_ASSESSMENT_NON_DIAGNOSIS",
        "pass_key": "FLOW_ASSESSMENT_PASSED",
        "downstream_entry_key": "ARRIVED_FURTHER_ASSESS",
        "service_key": "SERVICE_ASSESSMENT_COMPLETED",
    },
    "further_assessment": {
        "param": "pct_non_diag_at_further_assessment",
        "arrived_key": "ARRIVED_FURTHER_ASSESS",
        "exit_key": "EXIT_FURTHER_NON_DIAGNOSIS",
        "pass_key": "FLOW_FURTHER_ASSESS_PASSED",
        "downstream_entry_key": "FLOW_DIAGNOSIS_CONFIRMED",
        "service_key": "SERVICE_FURTHER_ASSESS_COMPLETED",
    },
}


def run_stage_boundary_suite(stage_configs, boundary_name, boundary_prob):
    """
    Run branch-probability boundary checks for each stage.

    Parameters
    ----------
    stage_configs : dict
        Mapping of pathway stages to counter names and parameter names.
    boundary_name : str
        Label printed in assertion messages.
    boundary_prob : float
        Boundary probability, either ``1.0`` or ``0.0``.

    Returns
    -------
    None
        Assertions raise errors when a boundary condition fails.
    """
    for stage, cfg in stage_configs.items():
        print(f"Testing stage ({boundary_name}): {stage}")
        experiment = Experiment(
            auditor=Audit(), **{cfg["param"]: boundary_prob}
        )
        test_results = single_run(experiment, rep=0, run_length=RUN_LENGTH)

        # Base calculations on patients reaching the decision point.
        base_key = cfg.get("service_key", cfg["arrived_key"])
        base_count = test_results[base_key]

        if boundary_prob == 1.0:
            # Triage uses arrivals; queued stages use completed services.
            stage_exits = test_results[cfg["exit_key"]]
            assert stage_exits == base_count, (
                f"FAIL ({stage}, {boundary_name}): exits "
                f"({stage_exits}) did not match base count "
                f"({base_count})"
            )
            assert test_results[cfg["pass_key"]] == 0, (
                f"FAIL ({stage}, {boundary_name}): pass-through "
                "should be zero"
            )
            # Queued stages can still finish earlier steps.
            if stage == "triage":
                assert test_results[cfg["downstream_entry_key"]] == 0, (
                    f"FAIL ({stage}, {boundary_name}): downstream "
                    "entry should be zero"
                )
        elif boundary_prob == 0.0:
            assert test_results[cfg["exit_key"]] == 0, (
                f"FAIL ({stage}, {boundary_name}): exits at this "
                "stage should be zero"
            )
            assert base_count > 0, (
                f"FAIL ({stage}, {boundary_name}): expected "
                "patients to reach the decision point"
            )
            pass_count = test_results[cfg["pass_key"]]
            assert pass_count == base_count, (
                f"FAIL ({stage}, {boundary_name}): pass-through "
                f"({pass_count}) did not match base count "
                f"({base_count})"
            )
        else:
            raise ValueError("Boundary probability must be either 1.0 or 0.0")


# Run the unified verification processes
run_stage_boundary_suite(stage_configs, "100% exits", 1.0)
run_stage_boundary_suite(stage_configs, "0% exits", 0.0)

print("\n" + "=" * 60)
print(" SUCCESS: All stage boundary suite verification checks passed!")
print("=" * 60)


Testing stage (100% exits): triage
Testing stage (100% exits): screening
Testing stage (100% exits): pre_assessment
Testing stage (100% exits): assessment
Testing stage (100% exits): further_assessment
Testing stage (0% exits): triage
Testing stage (0% exits): screening
Testing stage (0% exits): pre_assessment
Testing stage (0% exits): assessment
Testing stage (0% exits): further_assessment

 SUCCESS: All stage boundary suite verification checks passed!


## 14. Replication Function

One simulation run is only one possible sample path. The `multiple_runs` function repeats the model many times and returns a DataFrame containing the KPI results for each replication.

The function can run with fixed deterministic seeds, which is useful for reproducibility, or with fresh stochastic seeds, which is useful for exploring natural variation.


In [82]:
def multiple_runs(
    experiment,
    n_reps=N_REP,
    run_length=RUN_LENGTH,
    n_jobs=-1,
    use_fixed_seed=True,
):
    """
    Run independent simulation replications in parallel.

    Parameters
    ----------
    experiment : Experiment
        Baseline parameter container copied for each worker.
    n_reps : int, default N_REP
        Number of replications to execute.
    run_length : float, default RUN_LENGTH
        Simulation horizon in days for each replication.
    n_jobs : int, default -1
        Number of joblib workers to use.
    use_fixed_seed : bool, default True
        Whether each replication should use deterministic seeds.

    Returns
    -------
    pandas.DataFrame
        One row per replication with output counters and KPIs.
    """

    def _run_worker(rep_idx):
        """
        Execute one isolated replication worker.

        Parameters
        ----------
        rep_idx : int
            Replication index assigned by the parent loop.

        Returns
        -------
        dict
            KPI dictionary returned by ``single_run``.
        """
        # Create a completely isolated copy of the experiment infrastructure
        local_exp = copy.deepcopy(experiment)
        local_exp.use_fixed_seed = use_fixed_seed

        if use_fixed_seed:
            # Set deterministic stream profile: Base seed + replication number
            local_exp.set_random_no_set(rep_idx)
        else:
            # Seeds off: use fresh non-deterministic runtime entropy.
            import secrets
            fresh_entropy = secrets.randbits(31)
            local_exp.random_number_set = fresh_entropy
            local_exp.init_sampling()

        # Execute the isolated single run tracking profile
        return single_run(
            experiment=local_exp, rep=rep_idx, run_length=run_length
        )

    # Dispatch parallel processes safely across localized copies
    all_results = Parallel(n_jobs=n_jobs)(
        delayed(_run_worker)(rep_idx=rep) for rep in range(n_reps)
    )

    # Injection of replication identifier tracking indexes
    for rep, result in enumerate(all_results):
        result["Replication"] = rep
        result["Seeded_Execution"] = use_fixed_seed

    return pd.DataFrame(all_results)


## 15. Seeded and Unseeded Replication Comparison

This cell runs two batches of replications. The seeded batch checks repeatable model behaviour under controlled random streams. The unseeded batch shows how outputs vary when new stochastic paths are generated.

The comparison reports both replication-level results and aggregate mean and standard deviation values for selected KPIs. This helps judge whether the model is stable enough for scenario analysis.


In [83]:
# ============================================================
# V&V TEST SUITE: RANDOM NUMBER CONTROL VERIFICATION
# ============================================================


def compare_core_outputs(df_a, df_b):
    """
Compare key output metrics between two result DataFrames.

```
Parameters
----------
df_a : pandas.DataFrame
    First result set.
df_b : pandas.DataFrame
    Second result set.

Returns
-------
bool
    True if all selected metrics are identical.
    """

    cols = [
    "ARRIVED_TOTAL",
    "ACCESS_REFERRAL_TO_ASSESSMENT_RTT_DAYS",
    "ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS",
]

    return df_a[cols].equals(df_b[cols])

print("\n" + "=" * 70)
print(" RANDOM NUMBER CONTROL VERIFICATION")
print("=" * 70)

# ------------------------------------------------------------

# TEST 1: FIXED-SEED REPRODUCIBILITY

# ------------------------------------------------------------

print("\n[TEST 1] Fixed-Seed Reproducibility")

exp1 = Experiment(auditor=Audit())

df_seeded_run1 = multiple_runs(
experiment=exp1,
n_reps=6,
run_length=RUN_LENGTH,
use_fixed_seed=True,
)

exp2 = Experiment(auditor=Audit())

df_seeded_run2 = multiple_runs(
experiment=exp2,
n_reps=6,
run_length=RUN_LENGTH,
use_fixed_seed=True,
)

seeded_match = compare_core_outputs(
df_seeded_run1,
df_seeded_run2,
)

if seeded_match:
    print(
" -> SUCCESS: Fixed-seed replications are "
"perfectly reproducible."
)
else:
    print(
" -> FAILURE: Fixed-seed runs produced "
"different outputs."
)

# ------------------------------------------------------------

# TEST 2: STOCHASTIC INDEPENDENCE

# ------------------------------------------------------------

print("\n[TEST 2] Unseeded Stochastic Independence")

exp3 = Experiment(auditor=Audit())

df_unseeded_run1 = multiple_runs(
experiment=exp3,
n_reps=6,
run_length=RUN_LENGTH,
use_fixed_seed=False,
)

exp4 = Experiment(auditor=Audit())

df_unseeded_run2 = multiple_runs(
experiment=exp4,
n_reps=6,
run_length=RUN_LENGTH,
use_fixed_seed=False,
)

unseeded_match = compare_core_outputs(
df_unseeded_run1,
df_unseeded_run2,
)

if not unseeded_match:
    print(
" -> SUCCESS: Unseeded runs are "
"stochastically independent."
)
else:
    print(
" -> FAILURE: Unseeded runs unexpectedly "
"produced identical outputs."
)

# ------------------------------------------------------------

# TEST 3: REPLICATION VARIABILITY

# ------------------------------------------------------------

print("\n[TEST 3] Replication Variability Summary")

df_seeded_run1["Execution_Type"] = "SEEDED"
df_unseeded_run1["Execution_Type"] = "UNSEEDED"

df_compare = pd.concat(
[
df_seeded_run1,
df_unseeded_run1,
],
ignore_index=True,
)

columns_to_show = [
"Execution_Type",
"Replication",
"SEED_USED",
"ARRIVED_TOTAL",
"ACCESS_REFERRAL_TO_ASSESSMENT_RTT_DAYS",
"ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS",
]

print(
df_compare[columns_to_show]
.to_string(index=False)
)

# ------------------------------------------------------------

# TEST 3A: SEED AUDIT TRAIL

# ------------------------------------------------------------

print("\n[TEST 3A] Seed Audit Trail")

seed_cols = [
"Execution_Type",
"Replication",
"SEED_USED",
]

print(
df_compare[seed_cols]
.sort_values(
["Execution_Type", "Replication"]
)
.to_string(index=False)
)

# ------------------------------------------------------------

# TEST 3B: SEED SUMMARY

# ------------------------------------------------------------

print("\n[TEST 3B] Seed Summary")

print(
f"Default Seed = {DEFAULT_RND_SET}"
)

if len(df_seeded_run1) > 0:

    print(
        f"Seeded Range = "
        f"{df_seeded_run1['SEED_USED'].min()} "
        f"to "
        f"{df_seeded_run1['SEED_USED'].max()}"
    )



print(
f"Unique Unseeded Seeds = "
f"{df_unseeded_run1['SEED_USED'].nunique()}"
)

# ------------------------------------------------------------

# TEST 4: SUMMARY STATISTICS

# ------------------------------------------------------------

print("\n[TEST 4] Summary Statistics")

summary_stats = (
df_compare.groupby("Execution_Type")
.agg(
{
"ARRIVED_TOTAL": ["mean", "std"],
"ACCESS_REFERRAL_TO_ASSESSMENT_RTT_DAYS": [
"mean",
"std",
],
"ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS": [
"mean",
"std",
],
}
)
.round(2)
)

print(summary_stats)

# ------------------------------------------------------------

# FINAL RESULT

# ------------------------------------------------------------

print("\n" + "=" * 70)

if seeded_match and not unseeded_match:

    print(
        "FINAL RESULT: RANDOM NUMBER CONTROL "
        "VERIFIED [PASS]"
    )

else:

    print(
        "FINAL RESULT: RANDOM NUMBER CONTROL "
        "FAILED [INVESTIGATE]"
    )

print("=" * 70)
print("Verification Completed")
print("=" * 70)


 RANDOM NUMBER CONTROL VERIFICATION

[TEST 1] Fixed-Seed Reproducibility
 -> SUCCESS: Fixed-seed replications are perfectly reproducible.

[TEST 2] Unseeded Stochastic Independence
 -> SUCCESS: Unseeded runs are stochastically independent.

[TEST 3] Replication Variability Summary
Execution_Type  Replication  SEED_USED  ARRIVED_TOTAL  ACCESS_REFERRAL_TO_ASSESSMENT_RTT_DAYS  ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS
        SEEDED            0         42           6609                               90.159351                              91.340190
        SEEDED            1         43           6509                               89.238655                              90.317058
        SEEDED            2         44           6531                               78.621227                              80.660976
        SEEDED            3         45           6690                               91.027154                              92.526023
        SEEDED            4         46           671

## 16. V&V Test Suite Definitions

This section defines the formal verification and validation checks used to build confidence in the model.

### Verification: Did We Build The Model Correctly?

Verification focuses on internal correctness. These tests check that the code behaves as specified, that counters balance, and that random seeding works as intended.

The suite includes:

1. Seed control and reproducibility checks: fixed seeds should produce identical results for key outputs.
2. Stochastic independence checks: unseeded runs should be allowed to differ, showing that randomness is active.
3. Flow conservation checks: arrivals, exits, and patients remaining in the system should balance.
4. Mathematical convergence checks: simulated averages should be broadly consistent with the expected means of the input distributions.

### Validation: Are The Model Outputs Plausible?

Validation focuses on whether the model is credible for its intended use. In this notebook, validation is supported by checking that pathway outputs follow the expected direction under extreme settings and that operational KPIs such as waiting times, backlogs, and utilisation respond to demand and capacity constraints.

These tests do not prove the model is a perfect representation of a real service. They provide evidence that the logic is coherent, measurable, reproducible, and suitable for comparing pathway scenarios once the input assumptions have been agreed with domain experts.

### How To Interpret Failures

A failed assertion should be treated as a model specification issue. It means either the test expectation is wrong, the pathway logic is wrong, or the result counter being checked is not measuring the intended event. The failure message gives the starting point for investigation.


In [84]:
# ============================================================
# SYSTEM VERIFICATION & VALIDATION SUITE
# ============================================================


def print_header(title):
    """
    Print a standard V&V section header.

    Parameters
    ----------
    title : str
        Header text to display.

    Returns
    -------
    None
        Text is written to standard output.
    """

    print("\n" + "=" * 65)
    print(f" SUITE: {title}")
    print("=" * 65)


# ============================================================
# HELPER FUNCTIONS
# ============================================================


def triangular_mean_days(duration_triplet):
    """
    Return the mean triangular duration in days.

    Parameters
    ----------
    duration_triplet : sequence of float
        Optimistic, most-likely, and pessimistic durations in hours.

    Returns
    -------
    float
        Mean duration converted from hours to days.
    """

    return sum(duration_triplet) / 3.0 / 24.0


# ============================================================
# TEST 1: SEED AND DETERMINISM VERIFICATION
# ============================================================


def run_seed_verification():
    """
    Verify deterministic and stochastic seed behaviour.

    Returns
    -------
    None
        Assertions raise errors if deterministic runs diverge.
    """
    print_header("1. SEED CONTROL & REPRODUCIBILITY VERIFICATION")

    auditor = Audit()

    print("[RUNNING] Testing Determinism (Fixed Seed = ON)...")

    exp_fixed_1 = Experiment(
        auditor=auditor, use_fixed_seed=True, random_number_set=101
    )

    res_fixed_1 = single_run(exp_fixed_1, rep=0, run_length=365)

    exp_fixed_2 = Experiment(
        auditor=auditor, use_fixed_seed=True, random_number_set=101
    )

    res_fixed_2 = single_run(exp_fixed_2, rep=0, run_length=365)

    assert (
        res_fixed_1["ARRIVED_TOTAL"] == res_fixed_2["ARRIVED_TOTAL"]
    ), "CRITICAL ERROR: Determinism failed."

    assert (
        res_fixed_1["ACCESS_REFERRAL_TO_ASSESSMENT_RTT_DAYS"]
        == res_fixed_2["ACCESS_REFERRAL_TO_ASSESSMENT_RTT_DAYS"]
    ), "CRITICAL ERROR: RTT metrics diverged."

    print(
        " -> SUCCESS: Fixed seeds are " "100% deterministic and reproducible."
    )

    print("[RUNNING] Testing stochastic independence...")

    exp_rand_1 = Experiment(auditor=auditor, use_fixed_seed=False)

    res_rand_1 = single_run(exp_rand_1, rep=0, run_length=365)

    exp_rand_2 = Experiment(auditor=auditor, use_fixed_seed=False)

    res_rand_2 = single_run(exp_rand_2, rep=0, run_length=365)

    if res_rand_1["ARRIVED_TOTAL"] != res_rand_2["ARRIVED_TOTAL"]:

        print(" -> SUCCESS: Random sampling is independent.")

    else:

        print(" -> WARNING: Matching totals " "occurred by coincidence.")


# ============================================================
# TEST 2: PATIENT FLOW MASS CONSERVATION
# ============================================================


def run_flow_conservation_verification():
    """
    Verify patient mass balance across the pathway.

    Returns
    -------
    None
        An assertion raises an error if arrivals are not conserved.
    """
    print_header("2. PATIENT FLOW MASS-CONSERVATION VERIFICATION")

    auditor = Audit()

    exp = Experiment(
        auditor=auditor, use_fixed_seed=True, random_number_set=42
    )

    print(
        "[RUNNING] Evaluating Mass-Balance "
        "across a standard 5-Year Horizon..."
    )

    results = single_run(exp, rep=1, run_length=RUN_LENGTH)

    arrived = results["ARRIVED_TOTAL"]
    exited = results["EXIT_TOTAL"]
    trapped = results["IN_SYSTEM_END"]

    calculated_balance = exited + trapped

    print(f"  * Collected Arrived Metrics: {arrived}")

    print(f"  * Collected Exited Metrics: {exited}")

    print(f"  * Trapped Backlog Remaining: {trapped}")

    print(f"  * Exited + Trapped Total: {calculated_balance}")

    assert arrived == calculated_balance, f"LOGIC BUG: Patient leak detected."

    print(" -> SUCCESS: Mass balance verified.")


# ============================================================
# TEST 3: CALENDAR-AWARE MATHEMATICAL CONVERGENCE
# ============================================================

def run_math_convergence_verification():
    """
    Verify that the infinite-capacity model converges towards
    expected RTT behaviour.

    This test validates scheduler consistency rather than exact
    pathway outputs. The objective is to confirm that RTTs remain
    mathematically plausible when queueing is removed.

    Returns
    -------
    None
        Assertions raise errors when convergence tolerances are
        exceeded.
    """

    print_header(
        "3. CALENDAR-AWARE MATHEMATICAL CONVERGENCE"
    )

    auditor = Audit()

    inf_capacity_settings = {
        "staff_screening": 10000,
        "staff_pre_assessment": 10000,
        "staff_assessment": 10000,
        "staff_further_assessment": 10000,
        "staff_post_diag_clinical": 10000,
        "staff_post_diag_other": 10000,
        "staff_review": 10000,
    }

    print(
        "[RUNNING] Infinite-capacity scenario "
        "(queueing removed)..."
    )

    exp = Experiment(
        auditor=auditor,
        use_fixed_seed=True,
        random_number_set=77,
        **inf_capacity_settings,
    )

    results = single_run(
        exp,
        rep=0,
        run_length=365 * 2,
    )

    # =====================================================
    # PART A: REFERRAL -> ASSESSMENT
    # =====================================================

    assessment_pathway = [
        DURATION_SCREENING,
        DURATION_PRE_ASSESSMENT,
        DURATION_ASSESSMENT,
    ]

    assessment_service_days = sum(
        triangular_mean_days(stage)
        for stage in assessment_pathway
    )

    assessment_empirical_rtt = results[
        "ACCESS_REFERRAL_TO_ASSESSMENT_RTT_DAYS"
    ]

    # Effective scheduler overhead observed in simulation
    assessment_calendar_delay = (
        assessment_empirical_rtt
        - assessment_service_days
    )

    assessment_delay_per_stage = (
        assessment_calendar_delay
        / len(assessment_pathway)
    )

    print("\nASSESSMENT PATHWAY")

    print(
        f"  * Raw Care Duration: "
        f"{assessment_service_days:.5f} Days"
    )

    print(
        f"  * Observed Calendar Delay: "
        f"{assessment_calendar_delay:.5f} Days"
    )

    print(
        f"  * Delay Per Stage: "
        f"{assessment_delay_per_stage:.5f} Days"
    )

    print(
        f"  * Simulation RTT: "
        f"{assessment_empirical_rtt:.5f} Days"
    )

    # =====================================================
    # PART B: REFERRAL -> DIAGNOSIS
    # =====================================================

    diagnosis_pathway = [
        DURATION_SCREENING,
        DURATION_PRE_ASSESSMENT,
        DURATION_ASSESSMENT,
        DURATION_FURTHER_ASSESSMENT,
    ]

    diagnosis_service_days = sum(
        triangular_mean_days(stage)
        for stage in diagnosis_pathway
    )

    diagnosis_theoretical_rtt = (
        diagnosis_service_days
        + (
            assessment_delay_per_stage
            * len(diagnosis_pathway)
        )
    )

    diagnosis_empirical_rtt = results[
        "ACCESS_REFERRAL_TO_DIAGNOSIS_RTT_DAYS"
    ]

    diagnosis_delta = abs(
        diagnosis_theoretical_rtt
        - diagnosis_empirical_rtt
    )

    print("\nDIAGNOSIS PATHWAY")

    print(
        f"  * Raw Care Duration: "
        f"{diagnosis_service_days:.5f} Days"
    )

    print(
        f"  * Expected RTT: "
        f"{diagnosis_theoretical_rtt:.5f} Days"
    )

    print(
        f"  * Simulation RTT: "
        f"{diagnosis_empirical_rtt:.5f} Days"
    )

    print(
        f"  * Delta: "
        f"{diagnosis_delta:.5f}"
    )

    # =====================================================
    # VALIDATION CHECKS
    # =====================================================

    # Infinite-capacity model should have
    # a sensible scheduler delay.

    assert (
        0.0 < assessment_delay_per_stage < 2.0
    ), (
        "Scheduler delay outside "
        "expected range."
    )

    # Diagnosis RTT should be predictable from
    # the observed assessment scheduler behaviour.

    diagnosis_tolerance = 1.0

    assert (
        diagnosis_delta < diagnosis_tolerance
    ), (
        f"Diagnosis convergence failed "
        f"(delta={diagnosis_delta:.5f})"
    )

    print(
        "\n -> SUCCESS: Infinite-capacity "
        "scheduler validation passed."
    )

## 17. Run The V&V Automation Engine

This final V&V cell executes the verification functions and reports whether the model passes all automated checks. A complete pass means the notebook has satisfied the internal tests currently defined here.

### What Each Test Is Testing

| Test | What it checks | Why it matters |
| --- | --- | --- |
| Seed control and reproducibility | Runs the model twice with the same fixed seed and compares key outputs such as total arrivals and referral-to-assessment RTT. | If fixed seeds are working, repeated runs should give the same results. This makes debugging, marking, and scenario comparison reproducible. |
| Stochastic independence | Runs the model with fixed seeding turned off and checks whether outputs can differ between runs. | This confirms that the model can still represent random variation when deterministic control is not required. |
| Patient flow mass conservation | Checks that total arrivals equal total exits plus patients still in the system at the end of the run. | This verifies that patients are not accidentally lost or double counted as they move through the pathway. |
| Calendar-aware mathematical convergence | Runs a high-capacity scenario and compares simulated RTT values with simplified theoretical waiting-time expectations. | This checks that service-time distributions and calendar slot logic produce plausible waiting-time behaviour when queueing pressure is removed. |

### How To Interpret The Output

If all assertions pass, the model has passed the current verification suite. If an assertion fails, the printed failure message identifies the part of the model that needs investigation, such as random seed control, patient counting, or RTT logic.

The output should be read as supporting evidence for model quality, not as a replacement for expert review. Real-world validation still requires checking the assumptions, probabilities, capacities, and waiting-time behaviour against service knowledge or observed data.


In [85]:
# ============================================================
# VERIFICATION & VALIDATION (V&V) AUTOMATION ENGINE
# ============================================================
print("\n" + "#" * 65)
print("  SYSTEM VERIFICATION & VALIDATION (V&V) AUTOMATION ENGINE")
print("#" * 65)

try:
    run_seed_verification()
    run_flow_conservation_verification()
    run_math_convergence_verification()

    print("\n" + "=" * 65)
    print(" FINAL AUDIT RESULT: ALL SYSTEMS VERIFIED [100% PASS]")
    print("=" * 65 + "\n")

except AssertionError as error:
    print("\n" + "!" * 65)
    print(" AUDIT CRITICAL FAILURE: MODEL SPECIFICATION VIOLATION DETECTED")
    print("!" * 65)
    print(f"Exception details: {error}\n")



#################################################################
  SYSTEM VERIFICATION & VALIDATION (V&V) AUTOMATION ENGINE
#################################################################

 SUITE: 1. SEED CONTROL & REPRODUCIBILITY VERIFICATION
[RUNNING] Testing Determinism (Fixed Seed = ON)...
 -> SUCCESS: Fixed seeds are 100% deterministic and reproducible.
[RUNNING] Testing stochastic independence...
 -> SUCCESS: Random sampling is independent.

 SUITE: 2. PATIENT FLOW MASS-CONSERVATION VERIFICATION
[RUNNING] Evaluating Mass-Balance across a standard 5-Year Horizon...
  * Collected Arrived Metrics: 6509
  * Collected Exited Metrics: 5938
  * Trapped Backlog Remaining: 571
  * Exited + Trapped Total: 6509
 -> SUCCESS: Mass balance verified.

 SUITE: 3. CALENDAR-AWARE MATHEMATICAL CONVERGENCE
[RUNNING] Infinite-capacity scenario (queueing removed)...

ASSESSMENT PATHWAY
  * Raw Care Duration: 0.26042 Days
  * Observed Calendar Delay: 3.58798 Days
  * Delay Per Stage: 1.19599 Days
 